# Bot-IoT secondary v2 — full

Hiếu owns review and Kaggle execution. Code preparation only; package remains PREPARED_PENDING_REVIEW.
Pinned package manifest: dd4fdc2551df68f294968f38ca3ecced0c27ac8bc2d1e1ae22239b7d676bae7c.
Attach the unchanged package and code bundle. Read README/protocol. Install requirements on Kaggle before running if necessary.
No local classifier fitting, preprocessing refit, TEST performance evaluation, tuning or resource fallback.


In [ ]:
import hashlib
import importlib.util
import json
import sys
import tempfile
import zipfile
from pathlib import Path

BUNDLE_ID = "iotguard_bot_iot_secondary_v2_kaggle_v1"
INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
if not INPUT.is_dir() or not WORKING.is_dir():
    raise RuntimeError("Run this notebook on Kaggle only")
sys.dont_write_bytecode = True

def find_code_bundle(input_root, working_root):
    candidates = []
    for manifest_path in input_root.rglob("bundle_manifest.json"):
        raw = manifest_path.read_bytes()
        if json.loads(raw).get("bundle_id") == BUNDLE_ID:
            candidates.append((hashlib.sha256(raw).hexdigest(), "directory", manifest_path.parent))
    for archive_path in input_root.rglob("*.zip"):
        with zipfile.ZipFile(archive_path) as archive:
            if "bundle_manifest.json" not in archive.namelist():
                continue
            if sum(i.file_size for i in archive.infolist()) > 8 * 1024**2:
                raise ValueError("Code ZIP exceeds bounded size")
            raw = archive.read("bundle_manifest.json")
            if json.loads(raw).get("bundle_id") == BUNDLE_ID:
                candidates.append((hashlib.sha256(raw).hexdigest(), "zip", archive_path))
    identities = {item[0] for item in candidates}
    if len(identities) != 1:
        raise ValueError("Attach one code version; identical smoke source snapshots are allowed")
    _, kind, path = sorted(candidates, key=lambda c: (c[1] != "directory", str(c[2])))[0]
    if kind == "directory":
        return path.resolve()
    code_root = Path(tempfile.mkdtemp(prefix="bot_iot_secondary_code_", dir=working_root)).resolve()
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise ValueError("Duplicate ZIP members")
        for item in archive.infolist():
            target = (code_root / item.filename).resolve()
            if (not target.is_relative_to(code_root) or "\\" in item.filename
                    or ":" in item.filename):
                raise ValueError("Unsafe ZIP path")
            if item.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with target.open("xb") as output:
                    output.write(archive.read(item.filename))
    return code_root

code_root = find_code_bundle(INPUT, WORKING)
spec = importlib.util.spec_from_file_location(
    "secondary_runner", code_root / "kaggle_secondary_runner.py")
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
runner.check_bundle()
print("Verified bundle:", code_root)


In [ ]:
PACKAGE_DIRECTORY = None  # Auto-discover exactly one pinned manifest in /kaggle/input.
SMOKE_RECEIPT_PATH = None  # Set the exact passing Smoke run_status.json path.
if not SMOKE_RECEIPT_PATH:
    raise ValueError("Set SMOKE_RECEIPT_PATH from the passing matching smoke run")

In [ ]:
result = runner.run("full", package_directory=PACKAGE_DIRECTORY, smoke_receipt=SMOKE_RECEIPT_PATH)
print(json.dumps(result, indent=2))

Retain the returned run ZIP and archive SHA256. Review run_status.json, warnings, row/group metrics and model reload parity. Full requires passing Smoke; no TEST metrics. Do not edit data manifest/status.
